### Why this notebook needs a patched kernel

ipykernel forces a `SelectorEventLoop` on Windows, which cannot spawn
subprocesses -- so Playwright's Node driver dies with `NotImplementedError`.

This venv's `python3` kernelspec is pointed at `pw_kernel.py`, which starts the
kernel on a `ProactorEventLoop` instead. Nothing special is needed in the cells
below; just use the normal `Python 3 (ipykernel)` kernel.


In [ ]:
import json
from dataclasses import dataclass
from typing import Any

from deepagents import create_deep_agent
from sqlalchemy.sql.base import elements

from models import model

agent = create_deep_agent(model=model)

result = agent.invoke({"messages": [{"role": "user", "content": "What is an LLM?"}]})

print(result["messages"][-1].content)

In [ ]:
with open("scan-page.js", "r", encoding="utf-8") as f:
    scan_page_js = f.read()

In [ ]:
from playwright.async_api import async_playwright

playwright = await async_playwright().start()
browser = await playwright.chromium.launch(headless=False)
page = await browser.new_page()

await page.goto("https://backlogr.dev")
result = await page.evaluate(scan_page_js)

In [ ]:
result_json = json.loads(result)

In [ ]:
result_json

In [ ]:
from dataclasses import dataclass

@dataclass
class Element:
    index: int
    signature: str
    tag: str
    text: str
    attrs: str
    cx: float
    cy: float

In [ ]:
elements = []
for index, raw_element in enumerate(result_json["elements"]):
    element = Element(index=index, **raw_element)
    elements.append(element)

In [ ]:
from typing import Any

def get_element_by_index(index: int) -> Element | Any:
    for current_element in elements:
        if current_element.index == index:
            return current_element
    return None

In [ ]:
element = get_element_by_index(2)
await page.mouse.click(element.cx, element.cy)

In [ ]:
await browser.close()
await playwright.stop()
